# Qwen3-8B Physics QLoRA fine-tune

Train the **physics idea-provocation** LoRA on Qwen3-8B in 4-bit (NF4).  
Input: 600 arxiv-sourced (quant-ph / hep-th / gr-qc / cond-mat) provocation pairs.  
Output: LoRA adapter that provokes research-depth physics connections.

**Recommended runtime: A100** (40 GB) — comfortable headroom.  
T4 (16 GB) also works with the settings below (batch=2, grad_accum=8, gradient_checkpointing), but is slower and tighter on memory.

**Set runtime:** Runtime → Change runtime type → Hardware accelerator: A100 (or T4).

**Upload before running cell 3:**
- `data/arxiv_training_pairs.jsonl` from your local machine.

**Download at the end:**
- `qwen3-8b-physics-lora.zip` — LoRA adapter, ~160 MB.

Estimated wall time: ~20 min on A100, ~60 min on T4.

In [ ]:
# 1. Install deps.
!pip -q install "transformers>=4.45" "peft>=0.13" "trl>=0.12" "datasets>=2.21" "bitsandbytes>=0.43" "accelerate>=0.34"

In [ ]:
# 2. Verify GPU.
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0))
print('vram MB:', torch.cuda.get_device_properties(0).total_memory // 1024**2)
props = torch.cuda.get_device_properties(0)
print(f'compute capability: {props.major}.{props.minor}')
assert props.major >= 7, 'bitsandbytes 4-bit needs compute capability >= 7.5; this GPU is too old.'

In [ ]:
# 3. Upload arxiv_training_pairs.jsonl from your local machine.
from google.colab import files
from pathlib import Path
Path('data').mkdir(exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    if name.endswith('.jsonl'):
        Path('data/arxiv_training_pairs.jsonl').write_bytes(content)
        print(f'wrote data/arxiv_training_pairs.jsonl ({len(content):,} bytes)')
        break

In [ ]:
# 4. Sanity-check the data.
import json
from collections import Counter
rows = [json.loads(l) for l in open('data/arxiv_training_pairs.jsonl', encoding='utf-8')]
print(f'pairs: {len(rows)}')
domains = Counter(r['x_domain'] for r in rows)
print(f'domain distribution: {dict(domains)}')
splits = Counter(r['split'] for r in rows)
print(f'cross/same split: {dict(splits)}')
novelty_avg = sum(r['novelty_score'] for r in rows) / len(rows)
print(f'mean novelty score: {novelty_avg:.2f}')
print(f'\nfirst input:  {rows[0]["input"]!r}')
print(f'first output (chars): {len(rows[0]["output"])}')
print(f'sample output:\n{rows[0]["output"][:400]}...')

In [ ]:
# 5. Configure & train.
import json, os, torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL  = 'Qwen/Qwen3-8B'
OUTPUT_DIR  = '/content/output/qwen3-8b-physics-lora'
EPOCHS      = 3
LR          = 2e-4
LORA_R      = 32          # doubled vs 1.7B run — more capacity for physics depth
LORA_ALPHA  = 64
MAX_SEQ_LEN = 1024
BATCH_SIZE  = 2           # safe for both T4 (16 GB) and A100 (40 GB)
GRAD_ACCUM  = 8           # effective batch = 16

os.makedirs(OUTPUT_DIR, exist_ok=True)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # Qwen3-8B is bf16; fp16 compute triggers grad scaler crash
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)  # enables gradient checkpointing

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

rows = [json.loads(l) for l in open('data/arxiv_training_pairs.jsonl', encoding='utf-8')]
ds = Dataset.from_list([{'text': f"{r['input']}\n\n{r['output']}"} for r in rows])

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=1,
    bf16=True,    # matches bnb_4bit_compute_dtype=bfloat16; required on A100
    fp16=False,
    optim='paged_adamw_8bit',
    report_to='none',
    max_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=ds, processing_class=tokenizer)
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('saved to', OUTPUT_DIR)

In [ ]:
# 6. Quick smoke test.
# `model` is the PEFT-wrapped model from cell 5 — do NOT call PeftModel.from_pretrained
# on it (adapter-on-adapter bug from the 1.7B run).
# Two-step templating: apply_chat_template → string, then tokenizer → BatchEncoding,
# so we can use inputs['input_ids'].shape[1] for slicing without ambiguity.
model.eval()

physics_prompts = [
    "What's a non-obvious way to think about quantum decoherence?",
    "I'm stuck on the black hole information paradox. Help me see it differently.",
    "Topological order in condensed matter -- give me an unusual angle.",
]

for prompt in physics_prompts:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,   # suppress Qwen3 <think> tokens → clean output
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=350,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.pad_token_id,
    )
    prompt_len = inputs['input_ids'].shape[1]
    response = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    print(f'PROMPT: {prompt}')
    print(f'RESPONSE:\n{response}')
    print('---')

In [ ]:
# 7. Zip and download the adapter.
import shutil
shutil.make_archive('/content/qwen3-8b-physics-lora', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/qwen3-8b-physics-lora.zip')